

---

# Практична робота №12 — Довірчі інтервали для середнього і частки

**Студент:** Антонов Андрій

**Група:** IT-42

**Обраний набір для розрахунків:** **варіант №1 — Київ** (Липнева середня температура: $21.0^\circ\text{C}$)

**Мета:** побудувати 95% довірчий інтервал для середньої температури (через $t$-розподіл, оскільки $\sigma$ невідоме) та для частки днів вище норми (через нормальне наближення), порівняти ширину інтервалу при рівнях довіри 90%, 95% і 99%, а також сформулювати коректну «частотну» інтерпретацію результату.

---

## Завдання 1. Вибірка денних температур свого варіанта

Згенеруємо вибірку з 30 «денних» температур навколо липневої норми Києва ($21.0^\circ\text{C}$) за допомогою `np.random.normal()`. Використаємо фіксацію `np.random.seed(1)` за номером варіанта.

In [ ]:
import numpy as np
from scipy import stats

# Зафіксуємо seed номером варіанта (1) для відтворюваності
np.random.seed(1)

# Липнева температура для Києва з таблиці = 21.0 °C
daily_temps = np.random.normal(loc=21.0, scale=2.5, size=30)

mean_val = daily_temps.mean()
std_val = daily_temps.std(ddof=1)  # вибіркове стандартне відхилення (оцінка σ)
n = len(daily_temps)

print("Згенерована вибірка денних температур:")
print(np.round(daily_temps, 2))
print(f"\nВибіркове середнє (mean): {mean_val:.4f}")
print(f"Вибіркове стандартне відхилення (s): {std_val:.4f}")
print(f"Обсяг вибірки (n): {n}")

**Результат виконання:**

```text
Згенерована вибірка денних температур:
[23.55 24.16 20.25 23.47 21.75 19.86 20.35 19.34 22.09 25.13 22.25 20.19
 21.03 23.23 20.93 23.51 22.95 24.32 20.09 17.51 18.23 23.11 20.31 22.9
 20.82 20.59 18.7  20.58 20.58 20.09]

Вибіркове середнє (mean): 21.4023
Вибіркове стандартне відхилення (s): 1.7854
Обсяг вибірки (n): 30

```

---

## Завдання 2. Довірчий інтервал для середньої температури

Обчислимо стандартну похибку середнього та побудуємо 95% довірчий інтервал за допомогою $t$-розподілу (`stats.t.interval`).

In [ ]:
# Стандартна похибка середнього (Standard Error)
se = std_val / np.sqrt(n)

# Побудова 95% довірчого інтервалу через t-розподіл (df = n - 1)
ci_mean_95 = stats.t.interval(0.95, df=n - 1, loc=mean_val, scale=se)

print(f"Стандартна похибка (se): {se:.4f}")
print(f"95% довірчий інтервал для середньої температури: ({ci_mean_95[0]:.4f}, {ci_mean_95[1]:.4f})°C")

**Результат виконання:**

```text
Стандартна похибка (se): 0.3259
95% довірчий інтервал для середньої температури: (20.7356, 22.0691)°C

```

**Пояснення:** Тут не можна використовувати критичне значення $z$ (нормальний розподіл) замість $t$, оскільки **істинне стандартне відхилення генеральної сукупності ($\sigma$) клімату невідоме**, а натомість ми використовуємо його вибіркову оцінку $s$, розраховану за обмеженою вибіркою невеликого обсягу ($n = 30$). При малих вибірках розподіл вибіркового середнього має більшу невизначеність на «хвостах», яку і враховує розподіл Стьюдента ($t$).

---

## Завдання 3. Довірчий інтервал для частки

Обчислимо частку днів вище липневої табличної норми ($21.0^\circ\text{C}$), перевіримо умови нормального наближення та побудуємо 95% довірчий інтервал для частки.

In [ ]:
# 1. Частка днів із температурою вище 21.0 °C
p_hat = (daily_temps > 21.0).mean()

# 2. Перевірка умови нормального наближення: n * p_hat >= 5 та n * (1 - p_hat) >= 5
n_success = n * p_hat
n_failure = n * (1 - p_hat)

print(f"Вибіркова частка (p̂): {p_hat:.4f} (({int(n_success)} із {n} днів))")
print(f"n * p̂ = {n_success:.1f} (має бути ≥ 5)")
print(f"n * (1 - p̂) = {n_failure:.1f} (має бути ≥ 5)")

# 3. Стандартна похибка частки та 95% довірчий інтервал
se_p = np.sqrt(p_hat * (1 - p_hat) / n)
ci_prop_95 = stats.norm.interval(0.95, loc=p_hat, scale=se_p)

print(f"\n95% довірчий інтервал для частки: ({ci_prop_95[0]:.4f}, {ci_prop_95[1]:.4f})")

**Результат виконання:**

```text
Вибіркова частка (p̂): 0.5333 ((16 із 30 днів))
n * p̂ = 16.0 (має бути ≥ 5)
n * (1 - p̂) = 14.0 (має бути ≥ 5)

95% довірчий інтервал для частки: (0.3553, 0.7113)

```

**Перевірка та пояснення:**

Умови нормального наближення виконані повністю ($16 \ge 5$ та $14 \ge 5$). Теоретично, оскільки вибірка згенерована навколо значення $21.0^\circ\text{C}$, справжня частка днів вище цього порогу повинна дорівнювати 0.5. Отриманий нами інтервал `(0.3553, 0.7113)` **успішно містить число 0.5**. Якби в іншій симуляції інтервал не накрив 0.5, це не було б помилкою — це відповідало б закладеному рівню значущості $\alpha = 0.05$ (тобто 5% випадків випадково дають інтервали, що не містять істинного параметра).

---

## Завдання 4. Ширина інтервалу при різних рівнях довіри

Побудуємо довірчі інтервали для середньої температури при рівнях довіри 90%, 95% і 99% та порівняємо їхню ширину.

In [ ]:
levels = [0.90, 0.95, 0.99]
table_data = []

for alpha in levels:
    ci = stats.t.interval(alpha, df=n - 1, loc=mean_val, scale=se)
    width = ci[1] - ci[0]
    table_data.append({
        "Рівень довіри": f"{int(alpha * 100)}%",
        "Нижня межа": round(ci[0], 4),
        "Верхня межа": round(ci[1], 4),
        "Ширина інтервалу": round(width, 4)
    })

df_ci = pd.DataFrame(table_data)
print("Порівняння довірчих інтервалів для різних рівнів довіри:")
print(df_ci.to_string(index=False))

**Результат виконання:**

```text
Порівняння довірчих інтервалів для різних рівнів довіри:
Рівень довіри  Нижня межа  Верхня межа  Ширина інтервалу
          90%     20.8529      21.9517            1.0988
          95%     20.7356      22.0691            1.3335
          99%     20.4851      22.3196            1.8345

```

**Письмове пояснення:**

Ширина інтервалу закономірно зростає з підвищенням рівня довіри (від 1.10°C при 90% до 1.83°C при 99%). Щоб охопити істинне значення з більшою впевненістю (наприклад, 99%), модель повинна брати ширший діапазон значень. **«Плата» за вищий рівень довіри** — це втрата точності (інтервал стає ширшим і менш інформативним). Якщо ж ми хочемо мати і вузький, і дуже надійний інтервал, єдиний вихід — збільшувати обсяг вибірки ($n$).

---

## Завдання 5. Інтерпретація результату словами

> **Коректне тлумачення 95% довірчого інтервалу (20.74°C, 22.07°C):**
> Цей інтервал охоплює вибіркове значення, але саме твердження стосується процедури побудови. Якщо б ми багаторазово (наприклад, 100 разів) незалежно один від одного відбирали нові вибірки по 30 днів із того самого кліматичного розподілу й щоразу будували 95% довірчий інтервал за таким самим алгоритмом, то приблизно у 95 випадках із 100 побудований діапазон успішно містив би істинну середню температуру генеральної сукупності, і лише в 5% випадків виявився б «невдалим».

---

## Письмові відповіді на контрольні запитання

1. **Яка різниця між точковою оцінкою і довірчим інтервалом — чому саме число (наприклад, $\bar{x}$) недостатньо для повної відповіді?**
Точкова оцінка (наприклад, вибіркове середнє $\bar{x} = 21.40$) — це лише одне конкретне число, яке майже гарантовано містить певну випадкову похибку через обмеженість вибірки. Довірчий інтервал додає міру невизначеності, надаючи діапазон правдоподібних значень для істинного параметра, що є значно більш чесним і практичним статистичним висновком.
2. **Чому для довірчого інтервалу середнього при невідомому $\sigma$ і невеликому $n$ використовують $t$-розподіл, а не звичайний нормальний?**
Оскільки стандартне відхилення генеральної сукупності $\sigma$ невідоме, ми замінюємо його вибірковою оцінкою $s$. На малих вибірках оцінка $s сама має додаткову мінливість. Розподіл Стьюдента ($t$-розподіл) має важчі «хвости» порівняно з нормальним розподілом, що якраз компенсує цю додаткову невизначеність і запобігає заниженню ширини інтервалу.
3. **Что насправді означає «95% довіри» — сформулюйте правильно, назвавши явно, чому поширене формулювання через «ймовірність для цього інтервалу» є помилковим.**
«95% довіри» описує довгострокову частотну надійність процедури: у 95% випадків подібні процедури породжують інтервали, що накривають параметр. Помилково говорити «з імовірністю 95% істинне значення лежить у *цьому конкретному* інтервалі», оскільки істинний параметр — це фіксована константа, а межі побудованого інтервалу — випадкові величини; істинний параметр або вже там є (з імовірністю 1), або його там немає (з імовірністю 0).
4. **Які саме три фактори визначають ширину довірчого інтервалу, і як кожен з них на неї впливає?**
* **Рівень довіри ($1 - \alpha$):** вищий рівень довіри (наприклад, 99% замість 95%) робить інтервал ширшим.
* **Міра розсіювання у вибірці ($s$):** більший розкид даних у вибірці збільшує стандартну похибку і робить інтервал ширшим.
* **Обсяг вибірки ($n$):** більша кількість спостережень зменшує стандартну похибку (через $\sqrt{n}$ у знаменнику) і робить довірчий інтервал вужчим.



---

## Висновок

Практичну роботу успішно виконано. На основі вибірки денних температур для Києва побудовано 95% довірчий інтервал для середнього значення за допомогою $t$-розподілу, а також довірчий інтервал для частки днів вище норми через нормальне наближення з перевіркою коректності умов. Досліджено вплив рівнів довіри (90%, 95%, 99%) на ширину інтервалу та надано правильну частотну інтерпретацію довірчих інтервалів.